In [1]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ.pop("SPARK_HOME", None)
os.environ.pop("PYSPARK_PYTHON", None)
os.environ.pop("PYSPARK_DRIVER_PYTHON", None)

In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName("test") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 16:28:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-09 11:28:20--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-09T12%3A23%3A38Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-09T11%3A23%3A36Z&ske=2026-03-09T12%3A23%3A38Z&sks=b&skv=2018-11-09&sig=zNvSVzfNZ5y023H7XLcSiyNuYm7R8KPKFaAt3VMMetY%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MzA1OTMwMSwibmJmIjoxNzczMDU1NzAxLCJwYXRo

In [ ]:
!gzip -dc fhvhv_tripdata_2021-01.csv.gz

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,

gzip: stdout: Broken pipe


In [4]:
!wc -l fhvhv_tripdata_2021-01.csv

11908469 fhvhv_tripdata_2021-01.csv


In [8]:
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [9]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [7]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [7]:
import pandas as pd

df_pandas = pd.read_csv('head.csv')
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [10]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [11]:
from pyspark.sql import types

In [12]:
schema = types.StructType([
            types.StructField('hvfhs_license_num', types.StringType(), True), 
            types.StructField('dispatching_base_num', types.StringType(), True), 
            types.StructField('pickup_datetime', types.TimestampType(), True), 
            types.StructField('dropoff_datetime', types.TimestampType(), True), 
            types.StructField('PULocationID', types.IntegerType(), True), 
            types.StructField('DOLocationID', types.IntegerType(), True), 
            types.StructField('SR_Flag', types.StringType(), True)
        ])

In [13]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [14]:
df = df.repartition(24)

In [17]:
df.write.parquet('fhvhv/2021/01/')

26/03/09 13:48:39 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/09 13:48:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/09 13:48:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/09 13:48:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


In [15]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [16]:
from pyspark.sql import functions as F

In [17]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02882|2021-01-04 07:56:57|2021-01-04 08:02:23|         241|         241|   NULL|
|           HV0003|              B02872|2021-01-04 13:52:02|2021-01-04 13:58:38|         196|          83|   NULL|
|           HV0003|              B02882|2021-01-03 15:11:11|2021-01-03 15:30:16|         243|          24|   NULL|
|           HV0003|              B02884|2021-01-04 15:35:32|2021-01-04 15:52:02|         174|          18|   NULL|
|           HV0003|              B02879|2021-01-05 09:58:46|2021-01-05 10:09:35|         165|         155|   NULL|
|           HV0003|              B02871|2021-01-02 13:06:08|2021-01-02 13:23:38|

In [18]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [19]:
crazy_stuff('B02884')

's/b44'

In [20]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [21]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-04|  2021-01-04|         117|         138|
|  e/9ce| 2021-01-05|  2021-01-05|          91|          89|
|  e/b47| 2021-01-04|  2021-01-04|         164|         230|
|  e/9ce| 2021-01-03|  2021-01-03|          10|         219|
|  e/b3e| 2021-01-04|  2021-01-04|         235|         247|
|  e/b48| 2021-01-04|  2021-01-04|          69|          20|
|  e/b42| 2021-01-01|  2021-01-01|          51|         159|
|  a/b37| 2021-01-02|  2021-01-02|           4|         181|
|  e/b32| 2021-01-05|  2021-01-05|          79|         137|
|  e/9ce| 2021-01-01|  2021-01-01|          60|         167|
|  a/b31| 2021-01-02|  2021-01-02|          76|          76|
|  e/a39| 2021-01-03|  2021-01-03|         132|         107|
|  e/acc| 2021-01-02|  2021-01-02|          41|         244|
|  s/b44| 2021-01-04|  2

In [22]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003') \
  .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-03 13:31:12|2021-01-03 13:40:17|         221|         206|
|2021-01-01 00:56:21|2021-01-01 00:59:05|          76|         180|
|2021-01-01 09:09:28|2021-01-01 09:20:10|          20|           3|
|2021-01-01 09:46:44|2021-01-01 10:07:10|         216|         210|
|2021-01-05 02:53:10|2021-01-05 03:09:09|          21|         210|
|2021-01-01 11:20:55|2021-01-01 12:02:17|         123|          86|
|2021-01-01 18:58:46|2021-01-01 19:22:58|         256|         142|
|2021-01-05 09:59:53|2021-01-05 10:27:22|          42|         142|
|2021-01-03 17:45:24|2021-01-03 17:51:05|         248|         212|
|2021-01-02 23:33:29|2021-01-02 23:57:07|         158|          42|
|2021-01-03 00:32:40|2021-01-03 00:45:34|          45|         170|
|2021-01-04 04:49:44|2021-01-04 04:54:14|       

In [27]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [23]:
spark.stop()